# Recycling Image Classifier on SageMaker with Bedrock Feedback and Metric-Gated Retraining

This notebook trains a recycling image classifier, deploys it to a SageMaker real-time endpoint, captures low-confidence predictions, asks Amazon Bedrock Claude Sonnet 4.6 to label small low-confidence batches from a fixed class list, starts a SageMaker Pipeline retraining job, and redeploys only if the candidate model beats the current model on validation accuracy.

In [ ]:
import sys
!{sys.executable} -m pip uninstall -y sagemaker sagemaker-core sagemaker-train sagemaker-serve sagemaker-mlops
!{sys.executable} -m pip install --no-cache-dir "sagemaker>=2.245.0,<3"

In [ ]:
%pip install -q --upgrade -r requirements.txt


In [ ]:
import base64
import io
import json
import shutil
import sys
import tarfile
import time
from urllib.parse import urlparse

import boto3
import matplotlib.pyplot as plt
import numpy as np
import requests
import sagemaker
from datasets import load_dataset
from sagemaker import image_uris
from sagemaker.huggingface import HuggingFace, HuggingFaceModel
from sagemaker.model_monitor import DataCaptureConfig
from sagemaker.serializers import JSONSerializer
from sagemaker.deserializers import JSONDeserializer

sys.path.append('scripts')

session = sagemaker.Session()
boto_session = boto3.Session()
REGION = boto_session.region_name
ROLE = sagemaker.get_execution_role()
BUCKET = session.default_bucket()

print({'region': REGION, 'bucket': BUCKET, 'role': ROLE})


## EDA

The dataset is small enough for a course but has an interesting real-world story: an AI recycling sorter that improves from low-confidence production images.

In [ ]:
HF_DATASET_ID = "viola77data/recycling-dataset"
IMAGE_COLUMN = "image"
LABEL_COLUMN = "label"
TRAIN_SPLIT = "train"

raw = load_dataset(HF_DATASET_ID)
train_ds = raw[TRAIN_SPLIT]
label_names = train_ds.features[LABEL_COLUMN].names
label_ids = np.array(train_ds[LABEL_COLUMN])
counts = np.bincount(label_ids, minlength=len(label_names))
print(train_ds)
print(dict(zip(label_names, counts.tolist())))

In [ ]:
rng = np.random.default_rng(42)
sample_indices = rng.choice(len(train_ds), size=8, replace=False)
fig, axes = plt.subplots(2, 4, figsize=(14, 7))
for ax, idx in zip(axes.ravel(), sample_indices):
    row = train_ds[int(idx)]
    ax.imshow(row[IMAGE_COLUMN].convert("RGB"))
    ax.set_title(label_names[row[LABEL_COLUMN]])
    ax.axis("off")
plt.tight_layout()

fig, axes = plt.subplots(2, 4, figsize=(14, 7))
seen = set()
ax_iter = iter(axes.ravel())
for i in range(len(train_ds)):
    row = train_ds[i]
    cls = row[LABEL_COLUMN]
    if cls in seen:
        continue
    seen.add(cls)
    ax = next(ax_iter)
    ax.imshow(row[IMAGE_COLUMN].convert("RGB"))
    ax.set_title(label_names[cls])
    ax.axis("off")
    if len(seen) == min(8, len(label_names)):
        break
plt.tight_layout()

plt.figure(figsize=(12, 4))
plt.bar(label_names, counts)
plt.xticks(rotation=45, ha="right")
plt.title("Class distribution")
plt.tight_layout()

## Initial Training

The notebook first stages the Hugging Face dataset to S3. The training job then reads the base dataset from the SageMaker `dataset` input channel, which is closer to a production pattern than downloading the public dataset inside every training job.

In [ ]:
PROJECT = "recycling-classifier-course"
local_dataset_dir = "/tmp/recycling-hf-dataset"
dataset_s3_prefix = f"{PROJECT}/datasets/recycling-hf-dataset"

shutil.rmtree(local_dataset_dir, ignore_errors=True)
raw.save_to_disk(local_dataset_dir)
DATASET_S3_URI = session.upload_data(
    path=local_dataset_dir, bucket=BUCKET, key_prefix=dataset_s3_prefix
)
print(DATASET_S3_URI)

In [ ]:
TRAINING_INSTANCE_TYPE = "ml.g4dn.xlarge"
TRANSFORMERS_VERSION = "4.36.0"
PYTORCH_VERSION = "2.1.0"
PY_VERSION = "py310"

estimator = HuggingFace(
    entry_point="train.py",
    source_dir="scripts",
    role=ROLE,
    instance_count=1,
    instance_type=TRAINING_INSTANCE_TYPE,
    transformers_version=TRANSFORMERS_VERSION,
    pytorch_version=PYTORCH_VERSION,
    py_version=PY_VERSION,
    metric_definitions=[
        {"Name": "validation:accuracy", "Regex": "accuracy=([0-9\.]+)"}
    ],
)

estimator.fit(inputs={"dataset": DATASET_S3_URI}, wait=True)
print(estimator.model_data)

In [ ]:
PROJECT = "recycling-classifier-course"
ENDPOINT_NAME = f"{PROJECT}-endpoint"
CURRENT_METRICS_S3_URI = f"s3://{BUCKET}/{PROJECT}/model-registry/current_metrics.json"


def parse_s3_uri(uri):
    parsed = urlparse(uri)
    return parsed.netloc, parsed.path.lstrip("/")


def seed_current_metrics_from_model_artifact(
    model_data_s3_uri, metrics_s3_uri, endpoint_name
):
    s3_client = boto3.client("s3")
    bucket, key = parse_s3_uri(model_data_s3_uri)
    local_tar = "/tmp/recycling-current-model.tar.gz"
    s3_client.download_file(bucket, key, local_tar)
    metrics = {}
    with tarfile.open(local_tar, "r:gz") as tar:
        for member in tar.getmembers():
            if member.name.endswith("eval_metrics.json"):
                metrics = json.loads(tar.extractfile(member).read().decode("utf-8"))
                break
    metrics.update(
        {
            "model_data": model_data_s3_uri,
            "endpoint_name": endpoint_name,
            "seeded_at_epoch": int(time.time()),
        }
    )
    out_bucket, out_key = parse_s3_uri(metrics_s3_uri)
    s3_client.put_object(
        Bucket=out_bucket,
        Key=out_key,
        Body=json.dumps(metrics, indent=2).encode("utf-8"),
        ContentType="application/json",
    )
    return metrics


current_metrics = seed_current_metrics_from_model_artifact(
    estimator.model_data, CURRENT_METRICS_S3_URI, ENDPOINT_NAME
)
current_metrics

## Deploy Real-Time Endpoint

The endpoint returns a top label, confidence, and top-k alternatives. SageMaker data capture stores input/output payloads in S3 for monitoring and audit.

In [ ]:
PROJECT = "recycling-classifier-course"
ENDPOINT_NAME = f"{PROJECT}-endpoint"
DATA_CAPTURE_S3_URI = f"s3://{BUCKET}/{PROJECT}/monitoring/data-capture"
DEPLOY_INSTANCE_TYPE = "ml.m5.large"
TRANSFORMERS_VERSION = "4.37.0"
PYTORCH_VERSION = "2.1.0"
PY_VERSION = "py310"

model = HuggingFaceModel(
    model_data=estimator.model_data,
    role=ROLE,
    transformers_version=TRANSFORMERS_VERSION,
    pytorch_version=PYTORCH_VERSION,
    py_version=PY_VERSION,
    entry_point="inference.py",
    source_dir="scripts",
)

data_capture_config = DataCaptureConfig(
    enable_capture=True,
    sampling_percentage=100,
    destination_s3_uri=DATA_CAPTURE_S3_URI,
    capture_options=["REQUEST", "RESPONSE"],
)

predictor = model.deploy(
    initial_instance_count=1,
    instance_type=DEPLOY_INSTANCE_TYPE,
    endpoint_name=ENDPOINT_NAME,
    data_capture_config=data_capture_config,
)
predictor.serializer = JSONSerializer()
predictor.deserializer = JSONDeserializer()
print("endpoint:", ENDPOINT_NAME)

In [ ]:
import requests
import base64
import json

url = ""

img_bytes = requests.get(url).content
img_b64 = base64.b64encode(img_bytes).decode("utf-8")

result = predictor.predict({"image": img_b64})

if isinstance(result, list):
    result = result[0]

if isinstance(result, bytes):
    result = result.decode("utf-8")

result = json.loads(result)

print("\nPrediction")
print("=" * 40)
print(f"Predicted label : {result['predicted_label']}")
print(f"Label ID        : {result['predicted_label_id']}")
print(f"Confidence      : {result['confidence']:.2%}")
print(f"Image size      : {result['width']} x {result['height']}")

print("\nTop predictions")
print("=" * 40)

for i, pred in enumerate(result["top_predictions"], start=1):
    print(f"{i}. {pred['label']:<15} " f"{pred['probability']:.2%}")

In [ ]:
import json
import boto3

iam = boto3.client("iam")
sts = boto3.client("sts")

account_id = sts.get_caller_identity()["Account"]
role_name = "AmazonSageMaker-ExecutionRole-20260513T140482"
dashboard_name = "recycling-classifier-course-dashboard"

policy_document = {
    "Version": "2012-10-17",
    "Statement": [
        {
            "Effect": "Allow",
            "Action": "cloudwatch:PutDashboard",
            "Resource": f"arn:aws:cloudwatch::{account_id}:dashboard/{dashboard_name}"
        }
    ]
}

iam.put_role_policy(
    RoleName=role_name,
    PolicyName="AllowCourseCloudWatchDashboard",
    PolicyDocument=json.dumps(policy_document)
)


## Monitoring Dashboard

The endpoint writes captured payloads to S3. The API Lambda emits CloudWatch metrics for confidence, low-confidence count, and retraining starts.

In [ ]:
PROJECT = 'recycling-classifier-course'
ENDPOINT_NAME = f'{PROJECT}-endpoint'
DATA_CAPTURE_S3_URI = f's3://{BUCKET}/{PROJECT}/monitoring/data-capture'

cloudwatch = boto3.client('cloudwatch')
dashboard_name = f'{PROJECT}-dashboard'
dashboard_body = {
    'widgets': [
        {
            'type': 'metric',
            'width': 12,
            'height': 6,
            'properties': {
                'region': REGION,
                'title': 'API confidence and retraining metrics',
                'metrics': [
                    ['RecyclingClassifierCourse', 'Confidence', 'EndpointName', ENDPOINT_NAME, {'stat': 'Average'}],
                    ['.', 'LowConfidenceCount', '.', '.', {'stat': 'Sum'}],
                    ['.', 'RetrainingStarted', '.', '.', {'stat': 'Sum'}],
                ],
                'period': 60,
            },
        }
    ]
}
cloudwatch.put_dashboard(DashboardName=dashboard_name, DashboardBody=json.dumps(dashboard_body))
print('dashboard:', f'https://{REGION}.console.aws.amazon.com/cloudwatch/home?region={REGION}#dashboards:name={dashboard_name}')
print('data capture prefix:', DATA_CAPTURE_S3_URI)


## Manual Lambda/API Setup and Pipeline Registration

Create IAM roles, IAM policies, Lambda functions, and API Gateway manually. This notebook only registers the SageMaker Pipeline and prints the environment variables your manually-created API Lambda needs.

The deployment Lambda must use `scripts/lambda_deploy_model.py`. The API Lambda must use `scripts/lambda_api_handler.py`.

In [ ]:
PROJECT = 'recycling-classifier-course'
ENDPOINT_NAME = f'{PROJECT}-endpoint'
PIPELINE_NAME = f'{PROJECT}-pipeline'
LOW_CONFIDENCE_PREFIX = f'{PROJECT}/low-confidence'
DEPLOY_INSTANCE_TYPE = 'ml.m5.large'
TRANSFORMERS_VERSION = '4.37.0'
PYTORCH_VERSION = '2.1.0'
PY_VERSION = 'py310'
BEDROCK_MODEL_ID = 'global.anthropic.claude-sonnet-4-6'
CONFIDENCE_THRESHOLD = '0.65'
MIN_RETRAIN_ITEMS = '3'
DEPLOY_LAMBDA_ARN = ''
API_URL = ''

if DEPLOY_LAMBDA_ARN.startswith('REPLACE_WITH'):
    raise ValueError('Set DEPLOY_LAMBDA_ARN to your manually-created deployment Lambda ARN.')

inference_image_uri = image_uris.retrieve(
    framework='huggingface',
    region=REGION,
    version=TRANSFORMERS_VERSION,
    image_scope='inference',
    instance_type=DEPLOY_INSTANCE_TYPE,
    base_framework_version=f'pytorch{PYTORCH_VERSION}',
    py_version=PY_VERSION,
)

from pipeline_definition import get_pipeline
pipeline = get_pipeline(REGION, ROLE, BUCKET, DEPLOY_LAMBDA_ARN, inference_image_uri, scripts_dir='scripts')
pipeline.upsert(role_arn=ROLE)

api_lambda_environment = {
    'SAGEMAKER_ENDPOINT_NAME': ENDPOINT_NAME,
    'LOW_CONFIDENCE_BUCKET': BUCKET,
    'LOW_CONFIDENCE_PREFIX': LOW_CONFIDENCE_PREFIX,
    'PIPELINE_NAME': PIPELINE_NAME,
    'CONFIDENCE_THRESHOLD': CONFIDENCE_THRESHOLD,
    'MIN_RETRAIN_ITEMS': MIN_RETRAIN_ITEMS,
    'BEDROCK_MODEL_ID': BEDROCK_MODEL_ID,
    'ALLOWED_LABELS_JSON': json.dumps(label_names),
    'METRIC_NAMESPACE': 'RecyclingClassifierCourse',
}
print('pipeline:', PIPELINE_NAME)
print('deploy lambda:', DEPLOY_LAMBDA_ARN)
print('api lambda environment:')
print(json.dumps(api_lambda_environment, indent=2))
print('api url:', API_URL)
